# 第26章 蒙特卡洛模拟与数值方法——当解析解不存在时

> **动机先行**: BS 公式很美, 但它只覆盖了最简单的一类合约。现实中的衍生品五花八门: 亚式期权看的是**平均价格**, 回看期权记住**历史最低**, 障碍期权在乎**路径是否碰线**。这些收益取决于整条价格轨迹的结构几乎没有解析解, **蒙特卡洛模拟**成为唯一通用的定价引擎。它的合法性来自第10章的大数定律——把"期望"变成"平均", 只要样本够多, 平均值必然收敛到真值。而朴素模拟 $1/\sqrt{M}$ 的收敛速度太慢, **方差缩减技术**是量化工程师的核心武艺。
>
> **量化实战定位**: 蒙特卡洛是现代衍生品定价的工业主力 (奇异期权、XVA、组合风险模拟)。本章你将掌握两个实用技能: 用离散化 GBM 给路径依赖结构定价; 用对偶变量与控制变量把计算成本降低数倍。

---

## 26.1 动机: BS 公式管不到的地方

BS 公式的隐含前提非常苛刻: 收益只取决于**到期那一天**的价格。只要破坏任何一条, 解析解就岌岌可危:

| 结构 | 收益函数 | 依赖什么 |
|------|---------|---------|
| 欧式 Call | $\max(S_T-K,\ 0)$ | 只有终点价 |
| **亚式** Call | $\max(\bar{S}-K,\ 0)$, $\bar{S}$ 为均价 | 整条路径的平均 |
| **回看** Call (浮动行权) | $S_T - \min_t S_t$ | 全程最低点 |
| **障碍**期权 | 触碰边界才生效/失效 | 是否碰线 |

它们不是数学家的玩具: 大宗商品市场偏爱亚式期权 (均价更难被操纵, 套保者关心的本来就是一段时间的平均采购成本); 浮动行权回看期权天然匹配"卖在最高点"的需求; 障碍结构是结构性票据的常用积木。

定价它们的通用思路只有一条——回到风险中性定价的**一般形式** (第25章):

$$
V_0 = e^{-rT}\,E^{Q}\big[f(S_{t_0}, S_{t_1}, \dots, S_{t_n})\big]
$$

收益 $f$ 现在吃进整条路径。解析积分不再可行, 但"期望"这个概念本身给我们留了一扇门: **用大量随机抽样的平均值去逼近它**。这就是蒙特卡洛方法, 而它之所以合法, 正是第10章的大数定律。

## 26.2 蒙特卡洛定价的数学骨架

**它解决什么问题**: 解析公式只对欧式、单资产、收益仅取决于终值的结构有效。对于亚式、回看、障碍等路径依赖结构, 收益是整条路径的函数, 没有解析解。蒙特卡洛把"期望"变成"平均": 大量抽样后取均值, 大数定律保证收敛。

### 26.2.1 估计量: 一个可以被检验的随机变量

给路径抽样 $M$ 次, 定义蒙特卡洛估计量:

$$
\hat{V}_M = e^{-rT}\cdot\frac{1}{M}\sum_{i=1}^{M} f\big(\text{path}_i\big)
$$

大数定律保证 $\hat{V}_M \to V_0$ (依概率); 中心极限定理更进一步告诉我们**估计误差的分布**: $\hat{V}_M$ 近似服从正态分布, 中心是真值, 标准差为

$$
SE = e^{-rT}\,\frac{\mathrm{sd}(f)}{\sqrt{M}}
$$

这给了每个 MC 价格一个自带的"误差条"——报价不带标准误的量化工程师是不合格的。下面用实验验证这套理论的每一个数字:

**代码导读**: 本块分两段——(1) 用不同路径数 M 做蒙特卡洛定价欧式 Call, 与 BS 解析值对比, 验证误差确实按 $1/\sqrt{M}$ 缩水; (2) 把小样本实验重复 2000 次, 展示估计量本身的分布——中心是真值、宽度等于理论标准误。

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
true_c = bs_call(S0, K, T, r, sigma)

rng = np.random.default_rng(26)

print("=== 蒙特卡洛定价欧式 Call: 估计值随路径数的收敛 ===")
print(f"解析真值 C = {true_c:.4f}")
print(f"{'路径数 M':>10} | {'MC估计':>9} | {'标准误':>9} | {'与真值误差':>10}")
for M in [100, 1000, 10000, 100000]:
    Z = rng.standard_normal(M)
    ST = S0*np.exp((r-0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoff = np.exp(-r*T)*np.maximum(ST-K, 0)
    est = payoff.mean()
    se = payoff.std()/np.sqrt(M)
    print(f"{M:>10} | {est:>9.4f} | {se:>9.4f} | {abs(est-true_c):>12.4f}")

# 把小样本实验重复很多次, 观察"估计量本身"的分布
n_exp, M_small = 2000, 500
ests = []
for _ in range(n_exp):
    Z = rng.standard_normal(M_small)
    ST = S0*np.exp((r-0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    ests.append(np.exp(-r*T)*np.maximum(ST-K, 0).mean())
ests = np.array(ests)
se_theory = np.exp(-r*T)*np.sqrt(np.var(np.maximum(
    S0*np.exp((r-0.5*sigma**2)*T + sigma*np.sqrt(T)*rng.standard_normal(200000))-K, 0))/M_small)

print()
print(f"=== 把小样本实验重复 {n_exp} 次 (每次 M={M_small}) ===")
print(f"{n_exp} 个估计值的均值   = {ests.mean():.4f}  (应接近真值 {true_c:.4f})")
print(f"{n_exp} 个估计值的标准差 = {ests.std():.4f}")
print(f"理论标准误 sqrt(Var/M) = {se_theory:.4f}")
cover = np.abs(ests - true_c) <= 1.96*se_theory
print(f"落在 [真值±1.96倍SE] 的比例 = {cover.mean()*100:.1f}%  (正态理论值 95%)")

# 可视化: 估计量的经验分布 vs 理论正态
fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.hist(ests, bins=50, density=True, alpha=0.65, color='#BBDEFB',
        label=f'{n_exp} 次重复实验的估计值')
xs = np.linspace(ests.min()-0.1, ests.max()+0.1, 400)
ax.plot(xs, norm.pdf(xs, true_c, se_theory), 'r-', lw=2,
        label=f'理论正态 N({true_c:.2f}, {se_theory:.2f}²)')
ax.axvline(true_c, color='#333333', linestyle='--', lw=2,
           label=f'真值 {true_c:.4f}')
for kk in [-1.96, 1.96]:
    ax.axvline(true_c + kk*se_theory, color='#E91E63', linestyle=':', lw=1.5)
ax.set_xlabel(f'MC 估计值 (每次实验 M={M_small} 条路径)', fontsize=12)
ax.set_ylabel('概率密度', fontsize=12)
ax.set_title('估计量本身是随机变量: 分布中心=真值, 宽度=标准误', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 蒙特卡洛定价欧式 Call: 估计值随路径数的收敛 ===
解析真值 C = 10.4506
     路径数 M |      MC估计 |       标准误 |      与真值误差
       100 |   12.2763 |    1.7401 |       1.8257
      1000 |   10.5268 |    0.4486 |       0.0762
     10000 |   10.4704 |    0.1459 |       0.0198
    100000 |   10.4171 |    0.0464 |       0.0334

=== 把小样本实验重复 2000 次 (每次 M=500) ===
2000 个估计值的均值   = 10.4597  (应接近真值 10.4506)
2000 个估计值的标准差 = 0.6693
理论标准误 sqrt(Var/M) = 0.6562
落在 [真值±1.96倍SE] 的比例 = 94.9%  (正态理论值 95%)
```

![估计量的分布: 2000次重复实验的直方图与理论正态N(10.45, 0.66²)几乎重合, 94.9%落在±1.96倍标准误内](images/ch26_fig1_estimator_distribution.png)

**观察**:

1. **误差与标准误同阶**: M=100 时误差 1.83 落在 SE=1.74 的一个波动范围内; M=100000 时误差 0.033 远小于 SE... 不对, 与 SE=0.046 同阶。误差从不消失, 只是随 $\sqrt{M}$ 缩水。
2. **估计量是个随机变量**: 重复 2000 次实验, 得到的不是同一个数而是**一整片分布**——中心恰在真值, 宽度恰为理论标准误, 94.9% 落入 ±1.96 倍标准误 (理论 95%)。第10章的置信区间语言在这里完美兑现: 你报出的每个 MC 价格都自带 95% 置信区间。
3. **平方根惩罚**: 精度提高 10 倍需要 100 倍路径。这是本节最重要的工程结论, 也是下一节方差缩减存在的理由。

### 26.2.2 从终点抽样到路径抽样

欧式期权只需终值, 单步抽样即可精确。但路径依赖结构的收益吃整条轨迹, 必须逐日演化:

$$
\ln S_{t+\Delta t} = \ln S_t + \left(r - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\,Z_k, \qquad k = 1, \dots, n
$$

由于每一步都是精确的对数正态增量 (第24年的成果), 这里的"离散化"没有引入任何近似——只是把一年切成 250 步分别抽样。

## 26.3 路径依赖型衍生品定价

现在把引擎开到路径依赖结构上。一次路径模拟能同时喂饱所有结构 (每条路径算四个收益):

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

def bs_call(S, K, T, r, sigma):
    """欧式 Call 的 BS 解析价 (用作欧式行的自检锚点)"""
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

np.random.seed(2026)

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_steps = 250
M = 100000
dtg = T/n_steps

mu_step = (r - 0.5*sigma**2)*dtg          # 对数价格的每步漂移
sd_step = sigma*np.sqrt(dtg)              # 每步波动

Z = np.random.standard_normal((M, n_steps))
logP = np.log(S0) + np.cumsum(mu_step + sd_step*Z, axis=1)
logP = np.concatenate([np.full((M, 1), np.log(S0)), logP], axis=1)
P = np.exp(logP)

ST = P[:, -1]
avg_arith = P.mean(axis=1)                # 算术平均价
avg_geo = np.exp(logP[:, 1:].mean(axis=1))  # 几何平均价
min_path = P.min(axis=1)
max_path = P.max(axis=1)
B = 130.0                                  # 障碍线

payoff_euro = np.maximum(ST-K, 0)
payoff_asia_a = np.maximum(avg_arith-K, 0)
payoff_asia_g = np.maximum(avg_geo-K, 0)
payoff_lookback = ST - min_path            # 浮动行权回看
payoff_barrier = np.where(max_path < B, np.maximum(ST-K, 0), 0.0)

disc = np.exp(-r*T)
mc_euro = disc*payoff_euro.mean(); se_euro = disc*payoff_euro.std()/np.sqrt(M)
mc_asa = disc*payoff_asia_a.mean(); se_asa = disc*payoff_asia_a.std()/np.sqrt(M)
mc_asg = disc*payoff_asia_g.mean(); se_asg = disc*payoff_asia_g.std()/np.sqrt(M)
mc_lbk = disc*payoff_lookback.mean(); se_lbk = disc*payoff_lookback.std()/np.sqrt(M)
mc_bar = disc*payoff_barrier.mean(); se_bar = disc*payoff_barrier.std()/np.sqrt(M)

# 几何平均亚式期权的连续平均解析解 (验证锚点): ln G ~ N(m_g, v_g^2)
m_g = np.log(S0) + (r - 0.5*sigma**2)*T/2
v_g = np.sqrt(sigma**2 * T/3)
d1g = (m_g + v_g**2 - np.log(K))/v_g
d2g = d1g - v_g
an_asg = np.exp(-r*T)*(np.exp(m_g + v_g**2/2)*norm.cdf(d1g) - K*norm.cdf(d2g))

print("=== 路径依赖型衍生品定价 (风险中性蒙特卡洛) ===")
print(f"参数: S0={S0:.0f}, K={K:.0f}, T={T:.0f}年, r={r*100:.0f}%, sigma={sigma*100:.0f}%, "
      f"日频{n_steps}步, {M} 条路径")
print()
print(f"{'结构':<24} | {'MC价格':>8} | {'标准误':>7} | 相对欧式")
print("-" * 60)
rows = [
    ('欧式 Call (终点价)',      mc_euro, se_euro, f'{mc_euro/mc_euro:.2f}x  解析={bs_call(S0,K,T,r,sigma):.4f}'),
    ('算术平均亚式 Call',       mc_asa,  se_asa,  f'{mc_asa/mc_euro:.2f}x  无解析解'),
    ('几何平均亚式 Call',       mc_asg,  se_asg,  f'连续平均解析解={an_asg:.4f}'),
    ('浮动行权回看 Call',       mc_lbk,  se_lbk,  f'{mc_lbk/mc_euro:.2f}x  最贵'),
    ('向上敲出障碍 Call (B=130)', mc_bar, se_bar,  f'{mc_bar/mc_euro:.2f}x  更便宜'),
]
for name, est, se, note in rows:
    print(f"{name:<24} | {est:>8.4f} | {se:>7.4f} | {note}")

# 可视化: 左图-亚式的平均价; 右图-回看的全程最低点
np.random.seed(7)
M_show = 8
Zs = np.random.standard_normal((M_show, n_steps))
Pshow = np.exp(np.log(S0) + np.cumsum(mu_step + sd_step*Zs, axis=1))
Pshow = np.concatenate([np.full((M_show, 1), S0), Pshow], axis=1)
t_axis = np.arange(n_steps+1)/n_steps
colors = plt.cm.tab10(np.linspace(0, 1, M_show))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
ax = axes[0]
for i in range(M_show):
    ax.plot(t_axis, Pshow[i], lw=1, alpha=0.75, color=colors[i])
    ax.plot(t_axis, Pshow[i].cumsum()/np.arange(1, n_steps+2), lw=2.2,
            alpha=0.9, color=colors[i], linestyle='--')
ax.axhline(K, color='black', linestyle=':', linewidth=1)
ax.set_xlabel('时间 (年)', fontsize=12)
ax.set_ylabel('价格', fontsize=12)
ax.set_title('亚式期权看的是平均值: 虚线为各路径的累积平均价', fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
for i in range(M_show):
    ax.plot(t_axis, Pshow[i], lw=1, alpha=0.7, color=colors[i])
    ax.plot(t_axis, np.minimum.accumulate(Pshow[i]), lw=2, alpha=0.85,
            color=colors[i], linestyle='--')
    ax.scatter([t_axis[-1]], [Pshow[i].min()], s=42, color=colors[i],
               marker='v', zorder=5)
ax.set_xlabel('时间 (年)', fontsize=12)
ax.set_ylabel('价格', fontsize=12)
ax.set_title('回看期权记住全程最低点 (▼): 收益 = S_T − min(S)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 路径依赖型衍生品定价 (风险中性蒙特卡洛) ===
参数: S0=100, K=100, T=1年, r=5%, sigma=20%, 日频250步, 100000 条路径

结构                       |     MC价格 |     标准误 | 相对欧式
------------------------------------------------------------
欧式 Call (终点价)            |  10.4533 |  0.0466 | 1.00x  解析=10.4506
算术平均亚式 Call              |   5.7522 |  0.0252 | 0.55x  无解析解
几何平均亚式 Call              |   5.5581 |  0.0245 | 连续平均解析解=5.5468
浮动行权回看 Call              |  16.6168 |  0.0462 | 1.59x  最贵
向上敲出障碍 Call (B=130)      |   3.5557 |  0.0202 | 0.34x  更便宜
```

![路径依赖的直觉: 左图-八条路径与其累积平均价虚线(亚式), 右图-同样的路径与各自的历史最低点标记(回看)](images/ch26_fig2_path_dependent.png)

**观察**:

1. **引擎自检通过**: 欧式 MC 价格 10.4533 与解析解 10.4506 之差远小于标准误 0.0466——机器先证明自己没坏, 再去算没答案的题。
2. **价格的相对排序全部符合直觉**: 亚式最便宜 (0.55×, 平均化削掉了部分波动); 回看最贵 (1.59×, "买在最低点"是永远正确的特权, 特权要有对价); 向上敲出障碍更便宜 (0.34×, 卖方少担了一段上行风险)。
3. **几何平均是算术平均的"解析亲戚"**: 几何亚式有连续平均的封闭解 (5.5468), 我们的离散模拟给出 5.5581, 微小差距来自"250个采样点的平均"与"连续平均"的差异。算术平均 ≥ 几何平均 (均值不等式!), 所以算术亚式 (5.7522) 略贵于几何版——顺序完全正确。

> 💡 **为什么先算几何平均**: 它不只是教学彩蛋。实务中它是算术亚式期权最好的控制变量之一 (下一节), 也是解析近似 (如 Curran 1997) 的出发点。

## 26.4 方差缩减技术: 用聪明换算力

朴素 MC 的标准误 $\propto 1/\sqrt{M}$: 想要精度翻倍? 四倍算力。想要再翻倍? 十六倍。这条平方根定律在算力昂贵的年代催生了一门手艺——**方差缩减**: 构造均值不变、方差更小的估计量。介绍两种最常用的。

**对偶变量 (Antithetic Variates)**: 每个 $Z$ 都配上它的镜像 $-Z$。两者驱动的路径一涨一跌互为补偿, 若收益函数大体单调, 一对的平均就比单次抽样稳定得多。

**控制变量 (Control Variates)**: 找一个与我们关心的收益 $X$ 高度相关、且期望 $\mu_Y$ **精确已知的量** $Y$, 构造

$$
X_{cv} = X - \beta\,(Y - \mu_Y), \qquad \beta^* = \frac{\mathrm{Cov}(X, Y)}{\mathrm{Var}(Y)}
$$

$Y$ 的波动被减掉了, 只留下与 $X$ 共舞的部分。可以证明最优系数下 $\mathrm{Var}(X_{cv}) = (1-\rho^2)\,\mathrm{Var}(X)$——相关性越强, 免费的午餐越大。我们的场景里现成放着完美的 $Y$: **欧式期权的收益** (其期望就是 BS 公式!)。

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

np.random.seed(77)

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_steps, M = 250, 100000
mu_step = (r - 0.5*sigma**2)/n_steps
sd_step = sigma/np.sqrt(n_steps)

Z = np.random.standard_normal((M, n_steps))
logP = np.log(S0) + np.cumsum(mu_step + sd_step*Z, axis=1)
logP = np.concatenate([np.full((M, 1), np.log(S0)), logP], axis=1)
P = np.exp(logP)
ST = P[:, -1]
X = np.maximum(P.mean(axis=1)-K, 0)     # 目标: 算术平均亚式收益
Y = np.maximum(ST-K, 0)                 # 控制变量: 欧式收益 (期望精确已知)
EY = bs_call(S0, K, T, r, sigma)        # 欧式 Call 的解析期望

disc = np.exp(-r*T)
se_plain = disc*X.std()/np.sqrt(M)

# --- 方法一: 对偶变量 ---
Zp = np.random.standard_normal((M//2, n_steps))
l1 = np.log(S0)+np.cumsum(mu_step + sd_step*Zp, axis=1)
l1 = np.concatenate([np.full((M//2, 1), np.log(S0)), l1], axis=1)
l2 = np.log(S0)+np.cumsum(mu_step - sd_step*Zp, axis=1)
l2 = np.concatenate([np.full((M//2, 1), np.log(S0)), l2], axis=1)
pair = (np.maximum(np.exp(l1).mean(axis=1)-K, 0) +
        np.maximum(np.exp(l2).mean(axis=1)-K, 0))/2
se_anti = disc*pair.std(ddof=1)/np.sqrt(M//2)

# --- 方法二: 控制变量 (以欧式 Call 为控制, 其期望精确已知) ---
beta = np.cov(X, Y)[0, 1]/np.var(Y, ddof=1)
X_cv = X - beta*(Y - EY)
se_cv = disc*X_cv.std(ddof=1)/np.sqrt(M)
corr = np.corrcoef(X, Y)[0, 1]

print("=== 方差缩减: 以算术平均亚式 Call 为目标 ===")
print(f"朴素蒙特卡洛标准误 = {se_plain:.4f}  (基准)")
print()
print(f"{'方法':<22} | {'标准误':>7} | {'方差缩减':>8} | {'等效路径数':>9}")
print("-" * 58)
eff_anti = (se_plain/se_anti)**2
eff_cv = (se_plain/se_cv)**2
print(f"{'对偶变量 (Z 与 -Z)':<21} | {se_anti:>7.4f} | {eff_anti:>6.1f}x | {int(eff_anti*M):>9}")
print(f"{'控制变量 (欧式Call)':<20} | {se_cv:>7.4f} | {eff_cv:>6.1f}x | {int(eff_cv*M):>9}")
print()
print(f"控制变量的原理: 收益与欧式Call的相关系数 = {corr:.3f}")
print(f"回归系数 beta = Cov(X,Y)/Var(Y) = {beta:.3f}")
print("相关越高, 能借走的已知信息越多 —— 欧式价格是解析已知的, 亚式蹭它的光")

# 可视化: 左图-三种方法的标准误对比; 右图-控制变量的相关结构
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

methods = ['朴素MC', '对偶变量', '控制变量\n(欧式Call)']
ses = [se_plain, se_anti, se_cv]
bars = axes[0].bar(methods, ses, width=0.55,
                   color=['#999999', '#4CAF50', '#2196F3'], alpha=0.85)
for b_, s_ in zip(bars, ses):
    axes[0].text(b_.get_x()+b_.get_width()/2, s_+0.0004, f'{s_:.4f}',
                 ha='center', fontsize=11)
axes[0].set_ylabel('标准误', fontsize=12)
axes[0].set_title('同一目标 (亚式Call), 三种方法的标准误对比', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

idx = np.random.choice(M, 4000, replace=False)
axes[1].scatter(Y[idx], X[idx], s=5, alpha=0.25, color='#2196F3')
bline = np.linspace(Y[idx].min(), Y[idx].max(), 10)
axes[1].plot(bline, beta*bline + (X.mean()-beta*Y.mean()), 'r-', lw=2,
             label=f'回归线 (斜率 beta={beta:.3f})')
axes[1].set_xlabel('欧式 Call 收益 Y (期望精确已知)', fontsize=12)
axes[1].set_ylabel('亚式 Call 收益 X', fontsize=12)
axes[1].set_title(f'控制变量的逻辑: 相关系数={corr:.3f}, 知道Y就能修正X', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 方差缩减: 以算术平均亚式 Call 为目标 ===
朴素蒙特卡洛标准误 = 0.0254  (基准)

方法                     |     标准误 |     方差缩减 |     等效路径数
----------------------------------------------------------
对偶变量 (Z 与 -Z)         |  0.0176 |    2.1x |    206963
控制变量 (欧式Call)        |  0.0137 |    3.4x |    341281

控制变量的原理: 收益与欧式Call的相关系数 = 0.841
回归系数 beta = Cov(X,Y)/Var(Y) = 0.456
相关越高, 能借走的已知信息越多 —— 欧式价格是解析已知的, 亚式蹭它的光
```

![方差缩减效果: 左图-三种方法的标准误柱状对比(0.0254/0.0176/0.0137); 右图-亚式与欧式收益高度相关(ρ=0.84)的散点图与回归线](images/ch26_fig3_variance_reduction.png)

**观察**:

1. **两种技术都有效且互补**: 对偶变量 2.1 倍、控制变量 3.4 倍。"等效路径数"是最直观的翻译——控制变量让 10 万条路径干出了 34 万条的活。
2. **理论对账**: 相关系数 $\rho = 0.841$, 理论最大缩减 $(1-\rho^2)^{-1} = 3.42$ 倍, 与实测 3.4 倍严丝合缝——$\beta^*$ 公式的威力一目了然。
3. **对偶变量的适用边界**: 它依赖收益对随机源的单调性。亚式 Call 对 $Z$ 大体单调所以有效; 若换成收益不单调的结构 (如数字期权), 补偿机制会失效甚至反噬。**没有万能的技术, 只有对症的技术**。
4. 工业实践中还会叠加重要性抽样、条件蒙特卡洛等更多层手段, 以及把 $\beta$ 的估计不确定性一并管理。本章的方法已经覆盖了日常需求的主体。

## 26.5 工作流演示: 从历史数据到衍生品报价

最后把全书串成一条生产线: 第21章的时间序列、第24章的已实现波动率给出参数, 第25章的风险中性定价给出框架, 本章的引擎完成计算。

In [ ]:
import numpy as np
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(88)

print("=== 工作流演示: 从真实数据现场估计参数并定价 ===")
# 后复权日收益 -> 年化波动率 (勾稽第24章的已实现波动率口径)
_df26 = pd.read_csv('data/ifind_price_data.csv')
_px26 = _df26[_df26['thscode'] == '600519.SH'].sort_values('time')['close']
_r26 = np.log(_px26).diff().dropna()
sigma_hat = float(_r26.std()*np.sqrt(252))
r_assume = 0.02             # 假设无风险利率 2%
S0h, Kh, Th = 100.0, 100.0, 1.0
nh = 250
M = 100000

mu_h = (r_assume - 0.5*sigma_hat**2)/nh   # 每步漂移
sd_h = sigma_hat/np.sqrt(nh)              # 每步波动
Zh = np.random.standard_normal((M, nh))
logPh = np.log(S0h) + np.cumsum(mu_h + sd_h*Zh, axis=1)
logPh = np.concatenate([np.full((M, 1), np.log(S0h)), logPh], axis=1)
Ph = np.exp(logPh)
STh = Ph[:, -1]
avgh = Ph.mean(axis=1)
minh = Ph.min(axis=1)
disch = np.exp(-r_assume*Th)

print(f"输入: sigma={sigma_hat*100:.2f}% (来自历史数据), r={r_assume*100:.0f}% (假设), mu 不需要!")
print("(第25章的教训: 风险中性定价只用得到波动率)")
print()
print(f"{'结构 (K=S=100, T=1年)':<26} | {'MC价格':>8} | {'标准误':>7}")
print("-" * 50)
for name, po in [('欧式 Call', np.maximum(STh-Kh, 0)),
                 ('算术平均亚式 Call', np.maximum(avgh-Kh, 0)),
                 ('浮动行权回看 Call', STh-minh),
                 ('向下敲入 Put (B=85)', np.where(Ph.min(axis=1) <= 85,
                                                  np.maximum(Kh-STh, 0), 0))]:
    est = disch*po.mean()
    se = disch*po.std()/np.sqrt(M)
    print(f"{name:<26} | {est:>8.4f} | {se:>7.4f}")

**运行结果**:

```
=== 工作流演示: 从真实数据现场估计参数并定价 ===
输入: sigma=22.73% (来自历史数据), r=2% (假设), mu 不需要!
(第25章的教训: 风险中性定价只用得到波动率)

结构 (K=S=100, T=1年)         |     MC价格 |     标准误
--------------------------------------------------
欧式 Call                    |   9.9491 |  0.0499
算术平均亚式 Call                |   5.6705 |  0.0275
浮动行权回看 Call                |  17.0084 |  0.0506
向下敲入 Put (B=85)            |   7.4050 |  0.0353
```

**解读**: 注意欧式价格 (9.98) 比 25 章的 10.45 略低——因为这里的假设利率更低 ($r=2\% < 5\%$), 行权价贴现得更多。参数如何进入价格、哪个参数重要 ($\sigma$)、哪个参数无关 ($\mu$), 到这里应该已经内化为直觉。真实交易台的报价系统, 本质上就是这条流水线加上实时行情、波动率曲面和风控约束。

## 26.6 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

1. **风险中性定价的一般形式**: $V_0 = e^{-rT}\,E^{Q}[f(\text{path})]$——收益可以是任意路径依赖函数
2. **MC 估计量**: $\hat{V}_M = e^{-rT}\frac{1}{M}\sum_i f_i$, 大数定律保证收敛
3. **标准误**: $SE = e^{-rT}\dfrac{\mathrm{sd}(f)}{\sqrt{M}}$; 报价必须附带置信区间
4. **路径演化的精确增量**: $\ln S_{t+\Delta t} = \ln S_t + (r-\frac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t}\,Z_k$
5. **路径依赖收益**: 亚式 $\max(\bar{S}-K,0)$; 回看 $S_T - \min_t S_t$; 障碍期权按触碰条件生效/失效
6. **对偶变量**: 用 $(Z, -Z)$ 成对抽样取平均; 适合收益关于 $Z$ 近似单调的结构
7. **控制变量**: $X_{cv} = X - \beta^*(Y-\mu_Y)$, $\beta^* = \dfrac{\mathrm{Cov}(X,Y)}{\mathrm{Var}(Y)}$
8. **控制变量的理论上限**: $\mathrm{Var}(X_{cv}) = (1-\rho^2)\,\mathrm{Var}(X)$; 方差缩减倍数 $= (1-\rho^2)^{-1}$
9. **效率换算**: 方差缩小 $k$ 倍 $\equiv$ 达到同样精度只需 $1/k$ 的路径数

## 26.7 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 一般定价框架 | $e^{-rT}E^Q[f(\text{path})]$ | 覆盖一切路径依赖结构 |
| 大数定律 | 平均值收敛到期望 | 模拟定价的合法性来源 |
| 标准误 | $\propto 1/\sqrt{M}$ | 每个报价自带的误差条 |
| 路径模拟 | 精确对数正态增量逐日演化 | 亚式/回看/障碍的通用引擎 |
| 几何亚式锚点 | 有解析解的"亲戚" | 验证引擎 + 控制变量的双重角色 |
| 对偶变量 | 镜像冲击互相补偿 | 免费的方差削减 |
| 控制变量 | 借已知期望的 相关量修正 | 工业级精度的主力手段 |

**最后一句话**: 蒙特卡洛方法的哲学是把"不会积分"变成"会掷骰子"——用大数定律兑换解析困难。当你理解了估计量的分布、学会了借已知信息降方差, 你就拥有了给几乎所有衍生品定价的通用能力。至此, 从随机游走到奇异期权, 本书第六阶段的定价之旅画上句号。

## 26.8 练习题

### 数学推导

**题1——标准误与预算规划**:

(a) 证明 MC 估计量的方差 $\mathrm{Var}(\hat{V}_M) = e^{-2rT}\,\mathrm{Var}(f)/M$。

(b) 设某收益的标准差 (未贴现) 为 20 元。要在 95% 置信下把价格误差控制在 0.01 元以内 ($1.96\times SE \le 0.01$), 至少需要多少条路径?

(c) 若改用方差缩减使 $\mathrm{Var}(f)$ 降低到原来的 $1/9$, 同样精度需要多少条路径? 结合 $SE \propto M^{-1/2}$ 解释"等效路径数"的含义。

**题2——对偶变量的适用条件**:

设 $Z \sim N(0,1)$, 收益为 $f(Z)$。

(a) 证明若 $f$ 是单调递增函数, 则 $\mathrm{Cov}(f(Z), f(-Z)) \le 0$。(提示: 对固定的第一个坐标, 两变量关于彼此反向变动; 严格论证可用共单调性或切比雪夫联不等式的反号版本。)

(b) 由此说明对偶平均 $\frac{f(Z)+f(-Z)}{2}$ 的方差不超过 $\mathrm{Var}(f(Z))/2$。

(c) 举一个收益函数非单调的例子 (提示: 数字期权 $\mathbb{1}_{S_T>K}$ 在临界点附近跳变), 说明此时对偶变量的补偿机制为何可能失效。

**题3——控制变量的最优系数**:

(a) 对 $X_{cv}(\beta) = X - \beta(Y - \mu_Y)$, 求 $\beta$ 使 $\mathrm{Var}(X_{cv})$ 最小, 证明最优值为 $\beta^* = \mathrm{Cov}(X,Y)/\mathrm{Var}(Y)$。

(b) 证明对应的最小方差为 $(1-\rho^2)\,\mathrm{Var}(X)$, 其中 $\rho$ 为 $X, Y$ 的相关系数。

(c) 代入本章实测的 $\rho = 0.841$, 计算理论缩减倍数并与 26.4 节表格对照; 再算出当 $\rho$ 只有 0.5 时的缩减倍数, 说明"寻找高相关的控制变量"为什么值得花心思。

### 编程实践

**题1——障碍期权的离散监控偏差**: 连续障碍条款理论上要求"任何瞬间"不触碰边界, 而日频模拟只在收盘检查。对向上敲出障碍 Call ($K=100, B=130$), 分别用 $n \in \{50, 125, 250, 500, 1000\}$ 步模拟并观察敲出概率随步数的单调变化方向, 解释离散监控系统性低估还是高估敲出概率, 并说明实务中如何用"连续校正公式"弥补。

**题2——完整生产线的敏感性分析**: 参考 26.5 节的工作流, 固定 $\sigma$ 取第24章实测值、$r = 2\%$, 对 $K \in \{90, 100, 110\} \times T \in \{0.5, 1.0\}$ 的六种组合定价算术平均亚式 Call (建议叠加对偶+控制变量)。整理成 $3 \times 2$ 价格矩阵, 并回答: 行权价每上移 10 元, 亚式期权掉价多少? 期限缩短一半呢? 你的表格符合"期权价值随实值程度和期限递增"的一般规律吗?

## 26.9 参考文献

1. Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*. Springer.（金融蒙特卡洛的权威教材, 方差缩减与路径生成技术的系统论述）

2. Boyle, P. P. (1977). "Options: A Monte Carlo Approach." *Journal of Financial Economics*, 4(3), 323-338.（首次将蒙特卡洛方法系统引入期权定价的开山之作）

3. Kemna, A. G. Z., & Vorst, A. C. F. (1990). "A Pricing Method for Options Based on Average Asset Values." *Journal of Banking & Finance*, 14(1), 113-129.（几何平均亚式期权解析解的出处, 26.3节的验证锚点）

4. Longstaff, F. A., & Schwartz, E. S. (2001). "Valuing American Options by Simulation: A Simple Least-Squares Approach." *Review of Financial Studies*, 14(1), 113-147.（LSM 方法: 蒙特卡洛处理美式提前行权的里程碑, 本章框架的自然延伸）